# 04 Baselines And Component Checks

Reviewer-facing baseline/component evidence notebook for the revision package.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path('/content/ECG-RAMBA')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
_run_setup('git status --short --branch', check=False)


REQUIRED_REVISION_ARTIFACTS_FOR_04 = [
    'predictions/oof_final_ema_predictions.npz',
    'manifests/oof_final_ema_freeze_manifest.json',
    'metrics/calibration_ci_oof_final_ema_predictions.json',
    'manifests/oof_final_ema_prediction_run_manifest.json',
    'tables/oof_final_ema_class_summary.csv',
]
NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/hrv_only_oof_predictions.npz',
    'predictions/hrv_domain_oof_predictions.npz',
    'metrics/hrv_only_baseline_summary.json',
    'metrics/hrv_domain_classifier_summary.json',
    'metrics/hrv_domain_summary.csv',
    'metrics/hrv_robustness_component_check_summary.json',
    'metrics/robustness_summary.csv',
    'tables/table_hrv_only_class_metrics.csv',
    'tables/table_hrv_domain_status.csv',
    'tables/table_hrv_domain_classifier_confusion.csv',
    'tables/table_hrv_domain_classifier_fold_summary.csv',
    'tables/table_robustness.csv',
    'manifests/hrv_domain_analysis_manifest.json',
    'manifests/hrv_robustness_input_contract.json',
    'manifests/hrv_robustness_plan_alignment.json',
]
MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/minirocket_only_oof_predictions.npz',
    'metrics/minirocket_only_baseline_summary.json',
    'tables/table_minirocket_only_class_metrics.csv',
    'tables/table_minirocket_only_fold_summary.csv',
    'manifests/minirocket_only_baseline_manifest.json',
]
PAIRED_BASELINE_AVAILABLE_ARTIFACTS_FOR_04 = [
    'metrics/paired_full_vs_minirocket_comparison.json',
    'metrics/paired_full_vs_minirocket_bootstrap_samples.csv',
    'tables/table_paired_full_vs_minirocket.csv',
    'manifests/paired_full_vs_minirocket_manifest.json',
]
OPTIONAL_BASELINE_ARTIFACTS_FOR_04 = (
    NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04
    + MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04
    + PAIRED_BASELINE_AVAILABLE_ARTIFACTS_FOR_04
)


def _sha256_local(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _mirror_manifest_rows(mirror_root):
    manifest_path = Path(mirror_root) / 'manifests' / 'mirror_manifest.json'
    if not manifest_path.exists():
        return manifest_path, {}
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    declared_root = Path(payload.get('mirror_root', mirror_root))
    rows = {}
    for row in payload.get('artifacts', []):
        relative = row.get('relative_path')
        if not relative and row.get('mirror'):
            mirror_path = Path(row['mirror'])
            try:
                relative = mirror_path.relative_to(declared_root).as_posix()
            except ValueError:
                relative = mirror_path.name
        if relative:
            rows[Path(relative).as_posix()] = row
    return manifest_path, rows


def restore_required_revision_artifacts(required_rel_paths):
    import shutil

    destination_root = REPO_DIR / 'reports' / 'revision'
    manifest_path, rows = _mirror_manifest_rows(MIRROR_REVISION_ROOT)
    restored, mirror_path_copied, reused, unresolved = [], [], [], []
    for rel in required_rel_paths:
        rel = Path(rel).as_posix()
        destination = destination_root / rel
        row = rows.get(rel)
        needs_copy = not destination.exists()
        if row and destination.exists() and _sha256_local(destination) != row.get('sha256'):
            needs_copy = True
        if not needs_copy:
            reused.append(rel)
            continue

        source = MIRROR_REVISION_ROOT / rel
        copied = False
        if row and source.exists():
            expected_size = int(row.get('size_bytes', -1))
            if expected_size >= 0 and source.stat().st_size != expected_size:
                raise RuntimeError(f'Mirror size mismatch for required artifact: {rel}')
            if _sha256_local(source) != row.get('sha256'):
                raise RuntimeError(f'Mirror checksum mismatch for required artifact: {rel}')
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            if _sha256_local(destination) != row.get('sha256'):
                raise RuntimeError(f'Checksum mismatch after restoring required artifact: {rel}')
            restored.append(rel)
            copied = True
        if not copied and source.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            mirror_path_copied.append(rel)
            copied = True
            print(f'WARNING: copied {rel} from mirror path without a manifest row; downstream freeze SHA checks still apply.')
        if not copied:
            unresolved.append(rel)

    print(
        'Required revision artifact restore: '
        f'restored={len(restored)} mirror_path={len(mirror_path_copied)} reused={len(reused)} unresolved={len(unresolved)}'
    )
    if restored:
        print('Restored required artifacts:')
        for rel in restored:
            print(' -', rel)
    if mirror_path_copied:
        print('Copied required artifacts from mirror path without manifest rows; contract SHA checks still apply:')
        for rel in mirror_path_copied:
            print(' -', rel)
    if unresolved:
        print('WARNING: required artifacts were not found in the verified mirror manifest:', manifest_path)
        for rel in unresolved:
            print(' -', rel)


def restore_available_revision_artifacts(optional_rel_paths):
    import shutil

    destination_root = REPO_DIR / 'reports' / 'revision'
    manifest_path, rows = _mirror_manifest_rows(MIRROR_REVISION_ROOT)
    restored, reused, unavailable = [], [], []
    for rel in optional_rel_paths:
        rel = Path(rel).as_posix()
        source = MIRROR_REVISION_ROOT / rel
        row = rows.get(rel)
        if row and source.exists():
            expected_size = int(row.get('size_bytes', -1))
            if expected_size >= 0 and source.stat().st_size != expected_size:
                raise RuntimeError(f'Mirror size mismatch for available artifact: {rel}')
            source_sha = row.get('sha256')
            if _sha256_local(source) != source_sha:
                raise RuntimeError(f'Mirror checksum mismatch for available artifact: {rel}')
        elif source.exists():
            source_sha = _sha256_local(source)
        else:
            unavailable.append(rel)
            continue

        destination = destination_root / rel
        if destination.exists() and _sha256_local(destination) == source_sha:
            reused.append(rel)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        if _sha256_local(destination) != source_sha:
            raise RuntimeError(f'Checksum mismatch after restoring available artifact: {rel}')
        restored.append(rel)

    print(
        'Available revision artifact restore: '
        f'restored={len(restored)} reused={len(reused)} unavailable={len(unavailable)}'
    )
    if restored:
        print('Restored available revision artifacts from mirror:')
        for rel in restored:
            print(' -', rel)


if MIRROR_REVISION_ROOT.exists():
    restore_result = _run_setup(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        cwd=REPO_DIR,
        check=False,
    )
    if restore_result.returncode:
        print('WARNING: full checksum-verified mirror restore failed; trying selective restore for required inputs only.')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_04)
    restore_available_revision_artifacts(OPTIONAL_BASELINE_ARTIFACTS_FOR_04)
else:
    print('Mirror not found; Notebook 04 will require Notebook 02/03 artifacts in the active repo:', MIRROR_REVISION_ROOT)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
fingerprinted_rocket_caches = sorted(DRIVE_ROOT.glob('rocket_raw_N44186_C12_L5000_K10000_S42_R*.npz'))
print('  raw_minirocket_record_fingerprinted_count:', len(fingerprinted_rocket_caches))
for path in fingerprinted_rocket_caches[:5]:
    print(f'    - {path.name} size={path.stat().st_size}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Scope And Contracts

This notebook is intentionally CPU-friendly and artifact-first. It validates the frozen OOF and calibration outputs, writes reusable evidence tables, and records which fair baselines are complete or still blocked. It does not run full ECG-RAMBA training. It can optionally fit the lightweight MiniRocket-only classifier under the frozen OOF protocol, while unavailable model-specific runners remain blocked.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import math
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

if not Path.cwd().joinpath('configs', 'config.py').exists() and Path('/content/ECG-RAMBA/configs/config.py').exists():
    os.chdir('/content/ECG-RAMBA')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from scripts.revision.common import calibration_summary, git_commit, multilabel_metrics, save_csv, save_json, sha256_file

THRESHOLD = 0.5
N_BINS = 15
RUN_HEAVY_BASELINES = False

revision_root = Path('reports/revision')
OOF_STEM = 'oof_final_ema'
pred_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
run_manifest_path = revision_root / 'manifests' / f'{OOF_STEM}_prediction_run_manifest.json'
class_table_path = revision_root / 'tables' / f'{OOF_STEM}_class_summary.csv'

required_inputs = {
    'frozen_oof_predictions': pred_path,
    'oof_final_ema_freeze_manifest': freeze_path,
    'calibration_ci': calibration_path,
    'oof_run_manifest': run_manifest_path,
    'oof_class_summary': class_table_path,
}
for name, path in required_inputs.items():
    print(f'{name:24s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing and 'restore_required_revision_artifacts' in globals():
    print('Required inputs missing before contract validation; retrying selective artifact restore...')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_04)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    print('Required inputs still missing; attempting direct fallback restore from Drive roots.')
    drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
    mirror_root = drive_root / 'revision_artifacts' / 'reports' / 'revision'
    drive_repo_root = drive_root / 'ECG-RAMBA' / 'reports' / 'revision'
    fallback_roots = [mirror_root, drive_repo_root]
    restored_direct, unresolved_direct = [], []
    for destination in required_inputs.values():
        if destination.exists():
            continue
        rel = destination.relative_to(revision_root)
        source = next((root / rel for root in fallback_roots if (root / rel).exists()), None)
        if source is None:
            unresolved_direct.append(str(destination))
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        restored_direct.append(f'{source} -> {destination}')
    print(f'Direct fallback restore: restored={len(restored_direct)} unresolved={len(unresolved_direct)}')
    for item in restored_direct:
        print(' - restored', item)
    for item in unresolved_direct:
        print(' - unresolved', item)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Notebook 02/03 artifacts are required before notebook 04: ' + '; '.join(missing) +
        '. Run Notebook 04 from Setup, or verify the files exist under '
        '/content/drive/MyDrive/ECG-Ramba/revision_artifacts/reports/revision.'
    )

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before baseline/component checks.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 04 requires checkpoint_kind=final_ema in the freeze manifest.')

frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}

def verify_frozen_file(path, label):
    rel = path.as_posix()
    if rel not in frozen_artifacts:
        raise ValueError(f'Freeze manifest does not include {rel}')
    current_sha = sha256_file(path)
    if current_sha != frozen_artifacts[rel]['sha256']:
        raise RuntimeError(f'{label} SHA256 changed after OOF freeze: {rel}')
    return current_sha

pred_sha = verify_frozen_file(pred_path, 'Prediction')
run_manifest_sha = verify_frozen_file(run_manifest_path, 'Run manifest')
class_table_sha = verify_frozen_file(class_table_path, 'Class table')
freeze_sha = sha256_file(freeze_path)
if calibration.get('predictions_sha256') != pred_sha:
    raise RuntimeError('Calibration JSON does not match frozen OOF prediction SHA256.')
if calibration.get('freeze_manifest_sha256') != freeze_sha:
    raise RuntimeError('Calibration JSON does not match final_ema freeze manifest SHA256.')
expected_shape = {'y_prob': [freeze.get('validated_records'), freeze.get('n_classes')], 'y_true': [freeze.get('validated_records'), freeze.get('n_classes')]}
if calibration.get('shape') != expected_shape:
    raise RuntimeError(f'Calibration shape does not match freeze manifest: {calibration.get("shape")} != {expected_shape}')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'threshold': THRESHOLD,
    'n_bins': N_BINS,
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'run_manifest_sha256': run_manifest_sha,
    'class_table_sha256': class_table_sha,
    'freeze_manifest_sha256': freeze_sha,
    'calibration_dataset': calibration.get('dataset'),
    'calibration_protocol': calibration.get('protocol'),
    'calibration_shape': calibration.get('shape'),
}
save_json(revision_root / 'manifests' / 'baseline_component_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'baseline_component_input_contract.json')


## Revision Plan Alignment

The cells below bind notebook 04 to the tracked reviewer plan. They make incomplete fair baselines explicit instead of silently treating placeholders as evidence.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_BASE', 'EXP_CAL', 'EXP_POOL'])]
relevant_claims = claims[claims['claim_id'].isin(['C01', 'C02', 'C05', 'C06'])]
relevant_tasks = tasks[tasks['id'].isin(['A2', 'A3', 'A4', 'B1'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'baseline_component_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'baseline_component_plan_alignment.json')


## No-Skill Reference Baselines

These are sanity references computed from the frozen OOF labels only. They are not substitutes for the fair model baselines requested by reviewers, but they help interpret whether the frozen model beats trivial prevalence/constant references under the same metric functions.


In [ ]:
with np.load(pred_path, allow_pickle=False) as data:
    y_true = np.asarray(data['y_true'], dtype=np.float32)
    y_prob_full = np.asarray(data['y_prob'], dtype=np.float32)
    class_names = data['class_names'].astype(str).tolist() if 'class_names' in data.files else list(range(y_true.shape[1]))

prevalence = y_true.mean(axis=0).astype(np.float32)
reference_predictions = {
    'full_ecg_ramba_frozen_oof': y_prob_full,
    'prevalence_probability_reference': np.tile(prevalence, (len(y_true), 1)).astype(np.float32),
    'zero_probability_reference': np.zeros_like(y_true, dtype=np.float32),
    'constant_0_5_probability_reference': np.full_like(y_true, 0.5, dtype=np.float32),
}

rows = []
for name, y_prob in reference_predictions.items():
    metrics = multilabel_metrics(y_true, y_prob, threshold=THRESHOLD)
    cal = calibration_summary(y_true, y_prob, n_bins=N_BINS)
    rows.append({
        'model_or_reference': name,
        'dataset': 'chapman_oof',
        'n_records': int(y_true.shape[0]),
        'n_classes': int(y_true.shape[1]),
        'threshold': THRESHOLD,
        'evidence_role': 'frozen_model' if name == 'full_ecg_ramba_frozen_oof' else 'no_skill_reference_not_fair_baseline',
        'uses_evaluation_labels_to_set_probability': name == 'prevalence_probability_reference',
        'fair_baseline_claim_eligible': False,
        **metrics,
        **cal,
    })

reference_df = pd.DataFrame(rows)
reference_csv = revision_root / 'metrics' / 'no_skill_reference_baselines.csv'
reference_table = revision_root / 'tables' / 'table_no_skill_reference_baselines.csv'
reference_df.to_csv(reference_csv, index=False)
reference_df.to_csv(reference_table, index=False)
print('Wrote:', reference_csv)
print('Wrote:', reference_table)
display(reference_df)


## MiniRocket-only Baseline Runner

Optional fold-safe MiniRocket-only baseline. Set `RUN_MINIROCKET_ONLY_BASELINE=True` for the first run; later runs can reuse the mirrored artifact.


In [ ]:
RUN_MINIROCKET_ONLY_BASELINE = False

minirocket_command = (
    'python -u scripts/revision/10_minirocket_only_baseline.py '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --reuse-predictions --backend torch_linear --torch-epochs 20 --batch-size 4096 --stats-batch-size 1024 --lr 1e-3 --weight-decay 1e-4 --standardize train_fold --device cuda'
)
print('MiniRocket-only can now be run as a fold-safe feature baseline. It may take time because it fits fold-safe standardized linear heads on 20,000 cached features in A100 GPU batches.')
print('MiniRocket-only: implemented=', Path('scripts/revision/10_minirocket_only_baseline.py').exists(), 'planned_command=', minirocket_command)

if RUN_MINIROCKET_ONLY_BASELINE:
    run(minirocket_command, log_path='reports/revision/logs/baseline_minirocket_only.log')
else:
    print('MiniRocket-only execution disabled. Set RUN_MINIROCKET_ONLY_BASELINE=True in this cell to run it before the Fair Baseline Completion Matrix.')


## Fair Baseline Completion Matrix

This table is the reviewer-facing baseline ledger. A completed row must have the same split, preprocessing, threshold, and record-level protocol. Rows marked blocked must not be used to claim superiority over fair baselines.


In [ ]:
if 'restore_available_revision_artifacts' in globals():
    optional_artifacts_to_restore = globals().get('OPTIONAL_BASELINE_ARTIFACTS_FOR_04', NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04)
    if globals().get('RUN_MINIROCKET_ONLY_BASELINE', False):
        optional_artifacts_to_restore = [
            rel for rel in optional_artifacts_to_restore
            if rel not in globals().get('MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04', [])
        ]
        print('Skipping MiniRocket artifact restore from mirror because this notebook just generated them in the active repo.')
    restore_available_revision_artifacts(optional_artifacts_to_restore)

metric_names = [
    'roc_auc_macro', 'pr_auc_macro', 'f1_macro', 'f1_micro', 'precision_macro',
    'recall_macro', 'sensitivity_macro', 'specificity_macro', 'ppv_macro', 'npv_macro',
    'brier_macro', 'ece_macro', 'ece_max', 'mce_macro', 'mce_max'
]
full_metrics = {name: calibration.get('metrics', {}).get(name) for name in metric_names}
full_metrics.update({name: calibration.get('calibration', {}).get(name) for name in metric_names if name not in full_metrics or full_metrics[name] is None})
bootstrap_ci = calibration.get('bootstrap_ci', {})
ci_metric_map = {
    'roc_auc_macro': 'macro_roc_auc',
    'pr_auc_macro': 'macro_pr_auc',
    'f1_macro': 'f1_macro',
    'brier_macro': 'brier_macro',
    'ece_macro': 'ece_macro',
}

import shutil

def _read_summary_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        return {'_read_error': repr(exc)}

def _candidate_paths(relative_path):
    candidates = [('active_repo', revision_root / relative_path)]
    if 'MIRROR_REVISION_ROOT' in globals():
        candidates.append(('stable_drive_mirror', MIRROR_REVISION_ROOT / relative_path))
    if 'DRIVE_ROOT' in globals():
        candidates.append(('drive_repo', DRIVE_ROOT / 'ECG-RAMBA' / 'reports' / 'revision' / relative_path))
    return candidates

def _feature_summary_report(label, path, *, expected_feature_contract, expected_cache_kind_key, expected_protocol=None, expected_feature_preprocessing=None):
    path = Path(path)
    summary = _read_summary_json(path)
    load_info = summary.get('load_info', {}) if isinstance(summary, dict) else {}
    freeze_contract = load_info.get('freeze_contract') or {}
    expected_records = int(freeze.get('validated_records', -1))
    expected_classes = int(freeze.get('n_classes', -1))
    valid = (
        isinstance(summary, dict)
        and '_read_error' not in summary
        and summary.get('feature_contract') == expected_feature_contract
        and (expected_protocol is None or summary.get('protocol') == expected_protocol)
        and (expected_feature_preprocessing is None or summary.get('feature_preprocessing') == expected_feature_preprocessing)
        and int(summary.get('n_records', -1)) == expected_records
        and int(summary.get('n_classes', -1)) == expected_classes
        and summary.get('manuscript_ready', True) is True
        and int(load_info.get('oof_records_used', -1)) == expected_records
        and load_info.get('oof_predictions_sha256') == pred_sha
        and load_info.get(expected_cache_kind_key) == 'record_fingerprinted'
        and freeze_contract.get('checkpoint_kind') == 'final_ema'
    )
    return {
        'label': label,
        'path': path,
        'exists': path.exists(),
        'size': path.stat().st_size if path.exists() else None,
        'valid': valid,
        'summary': summary if isinstance(summary, dict) and '_read_error' not in summary else None,
        'read_error': summary.get('_read_error') if isinstance(summary, dict) else None,
        'feature_contract': summary.get('feature_contract') if isinstance(summary, dict) else None,
        'protocol': summary.get('protocol') if isinstance(summary, dict) else None,
        'feature_preprocessing': summary.get('feature_preprocessing') if isinstance(summary, dict) else None,
        'n_records': summary.get('n_records') if isinstance(summary, dict) else None,
        'n_classes': summary.get('n_classes') if isinstance(summary, dict) else None,
        'manuscript_ready': summary.get('manuscript_ready', True) if isinstance(summary, dict) else None,
        'oof_records_used': load_info.get('oof_records_used'),
        'oof_predictions_sha256': load_info.get('oof_predictions_sha256'),
        'cache_kind': load_info.get(expected_cache_kind_key),
        'freeze_checkpoint_kind': freeze_contract.get('checkpoint_kind'),
    }

def _select_feature_artifact(name, *, summary_relative_path, artifact_rel_paths, expected_feature_contract, expected_cache_kind_key, expected_protocol=None, expected_feature_preprocessing=None):
    reports = [
        _feature_summary_report(
            label,
            path,
            expected_feature_contract=expected_feature_contract,
            expected_cache_kind_key=expected_cache_kind_key,
            expected_protocol=expected_protocol,
            expected_feature_preprocessing=expected_feature_preprocessing,
        )
        for label, path in _candidate_paths(summary_relative_path)
    ]
    print(f'{name} artifact contract check:')
    print('  expected final_ema prediction sha256:', pred_sha)
    for report in reports:
        print(
            f"  {report['label']}: exists={report['exists']} valid={report['valid']} "
            f"size={report['size']} feature={report['feature_contract']} "
            f"protocol={report['protocol']} preprocessing={report['feature_preprocessing']} "
            f"n_records={report['n_records']} used={report['oof_records_used']} "
            f"manuscript_ready={report['manuscript_ready']} "
            f"oof_sha={report['oof_predictions_sha256']} cache_kind={report['cache_kind']} "
            f"freeze_kind={report['freeze_checkpoint_kind']} path={report['path']}"
        )
        if report['read_error']:
            print(f"    read_error={report['read_error']}")

    valid_reports = [report for report in reports if report['valid']]
    active_path = revision_root / summary_relative_path
    if valid_reports:
        chosen = valid_reports[0]
        if chosen['path'] != active_path:
            source_revision_root = chosen['path'].parents[1]
            copied = []
            for rel in artifact_rel_paths:
                source = source_revision_root / rel
                destination = revision_root / rel
                if source.exists():
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source, destination)
                    copied.append(rel)
            print(f'Restored canonical {name} artifacts into active repo from:', chosen['label'])
            print('  source revision root:', source_revision_root)
            print('  copied artifacts:', len(copied))
        return chosen['summary'], True, ''

    active_summary = reports[0]['summary']
    if active_summary is not None:
        candidate_notes = '; '.join(
            f"{report['label']}: exists={report['exists']}, valid={report['valid']}, "
            f"feature={report['feature_contract']}, protocol={report['protocol']}, "
            f"preprocessing={report['feature_preprocessing']}, n_records={report['n_records']}, "
            f"used={report['oof_records_used']}, manuscript_ready={report['manuscript_ready']}, "
            f"oof_sha={report['oof_predictions_sha256']}, cache_kind={report['cache_kind']}, "
            f"freeze_kind={report['freeze_checkpoint_kind']}"
            for report in reports
        )
        stale_reason = (
            f'Existing {name} summary is stale or non-canonical. '
            f'Expected final_ema oof_predictions_sha256={pred_sha}, '
            f'feature_contract={expected_feature_contract}, protocol={expected_protocol or "<any>"}, '
            f'feature_preprocessing={expected_feature_preprocessing or "<any>"}, '
            f'n_records={int(freeze.get("validated_records", -1))}, '
            f'n_classes={int(freeze.get("n_classes", -1))}, manuscript_ready=true, and record-fingerprinted cache. '
            f'Candidate status: {candidate_notes}.'
        )
        return active_summary, False, stale_reason
    return None, False, ''

hrv_only_summary_path = revision_root / 'metrics' / 'hrv_only_baseline_summary.json'
hrv_only_summary, hrv_only_artifact_valid, hrv_only_stale_reason = _select_feature_artifact(
    'HRV-only',
    summary_relative_path=Path('metrics/hrv_only_baseline_summary.json'),
    artifact_rel_paths=NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04,
    expected_feature_contract='hrv36',
    expected_cache_kind_key='chapman_hrv_cache_kind',
)

minirocket_summary_path = revision_root / 'metrics' / 'minirocket_only_baseline_summary.json'
minirocket_summary, minirocket_artifact_valid, minirocket_stale_reason = _select_feature_artifact(
    'MiniRocket-only',
    summary_relative_path=Path('metrics/minirocket_only_baseline_summary.json'),
    artifact_rel_paths=globals().get('MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04', []),
    expected_feature_contract='minirocket_raw',
    expected_cache_kind_key='minirocket_cache_kind',
    expected_protocol='minirocket_raw_standardized_torch_linear_same_folds_threshold_0.5',
    expected_feature_preprocessing='fold_train_standardization',
)

def baseline_row(name, family, status, evidence_path='', blocker='', notes='', metrics_payload=None, ci_payload=None, claim_reason=None):
    is_full = name == 'Full ECG-RAMBA frozen OOF'
    metrics_source = full_metrics if is_full else (metrics_payload or {})
    ci_source = bootstrap_ci if is_full else (ci_payload or {})
    row = {
        'baseline_name': name,
        'model_family': family,
        'dataset': 'chapman_oof',
        'split_protocol': 'same frozen five-fold OOF required',
        'threshold': THRESHOLD,
        'aggregation_protocol': 'record-level fixed threshold; Q=3 for full model where applicable',
        'status': status,
        'evidence_path': evidence_path,
        'blocker': blocker,
        'notes': notes,
        'claim_eligible_for_fair_baseline_superiority': False,
        'claim_eligibility_reason': claim_reason or ('Fair comparator rows are incomplete.' if is_full else 'Runner/result missing under the same protocol.'),
    }
    for metric in metric_names:
        row[metric] = metrics_source.get(metric, math.nan)
    for metric, ci_key in ci_metric_map.items():
        ci = ci_source.get(ci_key, {})
        row[f'{metric}_ci_low'] = ci.get('lo', math.nan)
        row[f'{metric}_ci_high'] = ci.get('hi', math.nan)
        row[f'{metric}_ci_mean'] = ci.get('mean', math.nan)
        row[f'{metric}_n_boot_valid'] = ci.get('n_boot_valid', math.nan)
    return row

def _completed_feature_row(name, summary, evidence_path, status, notes, claim_reason):
    return baseline_row(
        name,
        'feature_baseline',
        status,
        evidence_path=str(evidence_path),
        notes=notes,
        metrics_payload={**summary.get('metrics', {}), **summary.get('calibration', {})},
        ci_payload=summary.get('bootstrap_ci', {}),
        claim_reason=claim_reason,
    )

if hrv_only_artifact_valid:
    hrv_pending = 'Raw Mamba and ResNet1D/CNN' if minirocket_artifact_valid else 'Raw Mamba, ResNet1D/CNN, and MiniRocket-only'
    hrv_only_row = _completed_feature_row(
        'HRV-only',
        hrv_only_summary,
        hrv_only_summary_path,
        'complete_feature_baseline_from_notebook05',
        'Produced by scripts/revision/09_hrv_domain_analysis.py under the same frozen Chapman folds; report as HRV-only feature baseline, not full-model evidence.',
        f'HRV-only is complete, but fair-baseline superiority remains blocked until {hrv_pending} are complete.',
    )
elif hrv_only_summary is not None:
    hrv_only_row = baseline_row(
        'HRV-only', 'feature_baseline', 'stale_artifact_rejected',
        evidence_path=str(hrv_only_summary_path),
        blocker=hrv_only_stale_reason,
        notes='The existing HRV-only artifact is intentionally not used because it failed the canonical artifact contract.',
        claim_reason='Rerun Notebook 05 against oof_final_ema_predictions.npz, then rerun Notebook 04.',
    )
else:
    hrv_only_row = baseline_row(
        'HRV-only', 'feature_baseline', 'pending_notebook05_hrv_runner',
        blocker='Run Notebook 05 / scripts/revision/09_hrv_domain_analysis.py to produce hrv_only_baseline_summary.json.',
        notes='Required before claiming fair baseline superiority or HRV-only comparison.',
    )

if minirocket_artifact_valid:
    minirocket_row = _completed_feature_row(
        'MiniRocket-only',
        minirocket_summary,
        minirocket_summary_path,
        'complete_feature_baseline_from_script10',
        'Produced by scripts/revision/10_minirocket_only_baseline.py using record-fingerprinted RAW MiniRocket features, fold-train standardization, fixed-epoch linear logistic heads, and the frozen final_ema OOF folds.',
        'MiniRocket-only is complete, but fair-baseline superiority remains blocked until Raw Mamba and ResNet1D/CNN are complete.',
    )
elif minirocket_summary is not None:
    minirocket_row = baseline_row(
        'MiniRocket-only', 'feature_baseline', 'stale_artifact_rejected',
        evidence_path=str(minirocket_summary_path),
        blocker=minirocket_stale_reason,
        notes='The existing MiniRocket-only artifact is intentionally not used because it failed the canonical artifact contract.',
        claim_reason='Rerun scripts/revision/10_minirocket_only_baseline.py under Notebook 04, then rerun the baseline matrix cell.',
    )
else:
    minirocket_row = baseline_row(
        'MiniRocket-only', 'feature_baseline', 'pending_minirocket_runner',
        blocker='Run scripts/revision/10_minirocket_only_baseline.py from the Heavy Baseline Guard cell to produce minirocket_only_baseline_summary.json.',
        notes='Required before claiming fair baseline superiority or deterministic morphology-feature comparison.',
    )

baseline_rows = [
    baseline_row(
        'Full ECG-RAMBA frozen OOF',
        'full_model',
        'complete_frozen_oof',
        evidence_path=str(calibration_path),
        notes='Validated against oof_final_ema_freeze_manifest.json and calibration_ci_oof_final_ema_predictions.json.',
    ),
    baseline_row(
        'Raw Mamba', 'component_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority.',
    ),
    baseline_row(
        'ResNet1D/CNN', 'architecture_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority.',
    ),
    minirocket_row,
    hrv_only_row,
]

baseline_df = pd.DataFrame(baseline_rows)
baseline_csv = revision_root / 'metrics' / 'baseline_summary.csv'
baseline_table = revision_root / 'tables' / 'table_baselines.csv'
baseline_df.to_csv(baseline_csv, index=False)
baseline_df.to_csv(baseline_table, index=False)
print('Wrote:', baseline_csv)
print('Wrote:', baseline_table)
display(baseline_df)


## Paired Full vs MiniRocket Evidence

This cell turns the Full ECG-RAMBA vs MiniRocket-only baseline result into reviewer-ready paired evidence. It uses record-level paired bootstrap deltas, not independent CIs, so every resample compares both models on exactly the same records.


In [ ]:
RUN_PAIRED_FULL_VS_MINIROCKET = True
FORCE_RERUN_PAIRED_COMPARISON = False

paired_json = revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json'
paired_table = revision_root / 'tables' / 'table_paired_full_vs_minirocket.csv'
paired_samples = revision_root / 'metrics' / 'paired_full_vs_minirocket_bootstrap_samples.csv'
paired_manifest = revision_root / 'manifests' / 'paired_full_vs_minirocket_manifest.json'

paired_inputs = {
    'full_predictions': revision_root / 'predictions' / 'oof_final_ema_predictions.npz',
    'minirocket_predictions': revision_root / 'predictions' / 'minirocket_only_oof_predictions.npz',
    'freeze_manifest': revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json',
    'minirocket_summary': revision_root / 'metrics' / 'minirocket_only_baseline_summary.json',
    'minirocket_manifest': revision_root / 'manifests' / 'minirocket_only_baseline_manifest.json',
}
for label, path in paired_inputs.items():
    print(f'{label:24}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing_paired_inputs = [str(path) for path in paired_inputs.values() if not path.exists()]

from scripts.revision.common import sha256_file

def paired_output_stale():
    if not paired_json.exists() or not paired_table.exists() or not paired_manifest.exists():
        return True, 'paired output is missing'
    try:
        payload = json.loads(paired_json.read_text(encoding='utf-8'))
        inputs = payload.get('inputs', {})
        expected = {
            'full_predictions': inputs.get('full_predictions', {}).get('sha256'),
            'minirocket_predictions': inputs.get('comparator_predictions', {}).get('sha256'),
            'freeze_manifest': inputs.get('freeze_manifest', {}).get('sha256'),
            'minirocket_summary': inputs.get('minirocket', {}).get('summary', {}).get('sha256'),
            'minirocket_manifest': inputs.get('minirocket', {}).get('manifest', {}).get('sha256'),
        }
        current = {label: sha256_file(path) for label, path in paired_inputs.items()}
    except Exception as exc:
        return True, f'could not validate existing paired output: {exc!r}'
    mismatched = [label for label in expected if expected.get(label) != current.get(label)]
    if mismatched:
        return True, 'input SHA changed for: ' + ', '.join(mismatched)
    return False, 'existing paired output matches current input SHA values'

stale, stale_reason = paired_output_stale()
print('paired output stale:', stale, '|', stale_reason)

paired_command = (
    'python -u scripts/revision/11_paired_full_vs_minirocket.py '
    '--full-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--comparator-predictions reports/revision/predictions/minirocket_only_oof_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--comparator-summary reports/revision/metrics/minirocket_only_baseline_summary.json '
    '--comparator-manifest reports/revision/manifests/minirocket_only_baseline_manifest.json '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --seed 42 '
    '--expected-records 44186 --expected-classes 27 --expected-checkpoint-kind final_ema '
    '--require-manuscript-ready'
)
print('paired command:', paired_command)

if RUN_PAIRED_FULL_VS_MINIROCKET and missing_paired_inputs:
    raise FileNotFoundError('Paired comparison requires Notebook 02/03/04 artifacts: ' + '; '.join(missing_paired_inputs))
elif RUN_PAIRED_FULL_VS_MINIROCKET and (FORCE_RERUN_PAIRED_COMPARISON or stale):
    run(paired_command, log_path='reports/revision/logs/paired_full_vs_minirocket.log')
elif paired_json.exists():
    print('Skipping paired comparison because existing output passed input-SHA validation. Set FORCE_RERUN_PAIRED_COMPARISON=True to rerun.')
else:
    print('Paired comparison disabled. Set RUN_PAIRED_FULL_VS_MINIROCKET=True after MiniRocket-only artifacts exist.')

if paired_json.exists():
    payload = json.loads(paired_json.read_text(encoding='utf-8'))
    print('paired status:', payload.get('status'))
    print('paired output:', paired_json)
if paired_table.exists():
    paired_df = pd.read_csv(paired_table)
    display(paired_df[['metric', 'full_value', 'comparator_value', 'improvement_full_over_comparator', 'improvement_ci_low', 'improvement_ci_high', 'interpretation', 'safe_wording']])


## Component And Claim Evidence Ledger

This ledger connects the current artifacts to the claims that can and cannot be made after notebooks 02 and 03. It is designed for rebuttal traceability.


In [ ]:
pooling_sensitivity_path = revision_root / 'metrics' / 'pooling_sensitivity.csv'
pooling_decision_path = revision_root / 'metrics' / 'pooling_decision_summary.json'
paired_comparison_path = revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json'
paired_table_path = revision_root / 'tables' / 'table_paired_full_vs_minirocket.csv'
paired_available = paired_comparison_path.exists() and paired_table_path.exists()
if paired_available:
    paired_payload = json.loads(paired_comparison_path.read_text(encoding='utf-8'))
    paired_artifacts = '; '.join([str(paired_comparison_path), str(paired_table_path)])
else:
    paired_payload = None
    paired_artifacts = str(paired_comparison_path)
if pooling_sensitivity_path.exists() and pooling_decision_path.exists():
    pooling_decision = json.loads(pooling_decision_path.read_text(encoding='utf-8'))
    c05_status = 'supported_by_pooling_sensitivity_with_tradeoff_wording'
    c05_artifacts = '; '.join([str(pooling_sensitivity_path), str(pooling_decision_path)])
    c05_wording = pooling_decision.get('safe_wording', 'Report pooling sensitivity table and avoid optimality wording unless supported.')
else:
    pooling_decision = None
    c05_status = 'pending_pooling_sensitivity'
    c05_artifacts = str(pooling_sensitivity_path)
    c05_wording = 'Run notebook 06 pooling sensitivity before justifying Q=3 beyond the frozen OOF contract.'

required_fair_baselines = ['Raw Mamba', 'ResNet1D/CNN', 'MiniRocket-only', 'HRV-only']
complete_statuses = {'complete_feature_baseline_from_notebook05', 'complete_feature_baseline_from_script10', 'complete_frozen_oof'}
baseline_status = dict(zip(baseline_df['baseline_name'], baseline_df['status']))
missing_fair_baselines = [
    name for name in required_fair_baselines
    if baseline_status.get(name) not in complete_statuses
]
if missing_fair_baselines:
    c01_status = 'blocked_fair_baselines_missing_with_paired_minirocket_available' if paired_available else 'blocked_fair_baselines_missing'
    c01_wording = 'Do not claim broad fair-baseline superiority until these rows are complete: ' + ', '.join(missing_fair_baselines) + '.'
    if paired_available:
        c01_wording += ' Use the paired Full-vs-MiniRocket table for the narrower completed comparator: MiniRocket ranks better if PR-AUC/ROC-AUC CIs favor MiniRocket, while Full may be better at the fixed-threshold/calibrated operating point.'
else:
    c01_status = 'fair_baseline_rows_complete_with_paired_comparison' if paired_available else 'fair_baseline_rows_complete_pending_statistical_comparison'
    c01_wording = 'Only claim superiority for metrics whose paired bootstrap delta supports it.' if paired_available else 'Fair baseline rows are complete; only claim superiority after paired uncertainty/delta analysis supports it.'

component_rows = [
    {
        'claim_id': 'C01',
        'claim_topic': 'fair baseline superiority',
        'current_status': c01_status,
        'supporting_artifacts': '; '.join([str(revision_root / 'metrics' / 'baseline_summary.csv'), paired_artifacts]),
        'safe_wording': c01_wording,
    },
    {
        'claim_id': 'C02',
        'claim_topic': 'fixed-threshold ranking-decision gap',
        'current_status': 'supported_by_calibration_artifacts_with_limitations',
        'supporting_artifacts': '; '.join([
            str(calibration_path),
            str(revision_root / 'figures' / 'reliability_oof_final_ema_predictions.png'),
            str(revision_root / 'tables' / 'reliability_bins_oof_final_ema_predictions.csv'),
        ]),
        'safe_wording': 'Frozen OOF shows non-random ranking signal but low fixed-threshold recall/F1 and imperfect calibration; avoid clinical readiness wording.',
    },
    {
        'claim_id': 'C05',
        'claim_topic': 'Q=3 pooling operating point',
        'current_status': c05_status,
        'supporting_artifacts': c05_artifacts,
        'safe_wording': c05_wording,
    },
    {
        'claim_id': 'C06',
        'claim_topic': 'protocol-faithful OOF evaluation',
        'current_status': 'partial_oof_supported',
        'supporting_artifacts': '; '.join([str(freeze_path), str(run_manifest_path), str(calibration_path)]),
        'safe_wording': 'OOF protocol is frozen and traceable; external outputs remain experimental until separate readiness gates pass.',
    },
]
component_df = pd.DataFrame(component_rows)
component_table = revision_root / 'tables' / 'table_component_checks.csv'
component_json = revision_root / 'metrics' / 'component_check_summary.json'
component_df.to_csv(component_table, index=False)
save_json(component_json, {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'source_pooling_decision': str(pooling_decision_path) if pooling_decision_path.exists() else None,
    'pooling_decision': pooling_decision,
    'claims': component_rows,
})
print('Wrote:', component_table)
print('Wrote:', component_json)
display(component_df)


## Heavy Baseline Guard

The remaining model-specific fair baseline runners are not implemented in this branch. This cell records their intended command surface and stops accidental placeholder execution.


In [ ]:
planned = {
    'ResNet1D/CNN': 'python scripts/revision/TBD_resnet1d_baseline.py',
    'Raw Mamba': 'python scripts/revision/TBD_raw_mamba_baseline.py',
}
print('MiniRocket-only has a dedicated runner cell earlier in this notebook.')
print('HRV-only is intentionally excluded from this heavy-runner guard; Notebook 04 only imports its canonical artifact from Notebook 05.')

for name, command in planned.items():
    script = Path(command.split()[1])
    print(f'{name:16s}: implemented={script.exists()} planned_command={command}')

if RUN_HEAVY_BASELINES:
    missing = [name for name, command in planned.items() if not Path(command.split()[1]).exists()]
    if missing:
        raise RuntimeError('Model-specific fair baseline runners are not implemented yet: ' + ', '.join(missing))
    for name, command in planned.items():
        run(command, log_path=f'reports/revision/logs/baseline_{name.lower().replace("/", "_").replace(" ", "_")}.log')
else:
    print('Model-specific heavy baseline execution disabled. Current notebook writes traceable evidence and blocked ledger rows for unavailable runners.')


## Inventory And Stable Mirror

After notebook 04 writes evidence tables, freeze the revision artifact inventory and publish the artifacts to the Drive mirror for reuse.


In [ ]:
import inspect
if 'log_path' not in inspect.signature(run).parameters:
    raise RuntimeError('Notebook command helper run() was overwritten. Rerun the Setup cell before this inventory/mirror cell.')

run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/baseline_component_artifact_inventory.log',
)

mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/baseline_component_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'baseline_component_input_contract.json',
    revision_root / 'manifests' / 'baseline_component_plan_alignment.json',
    revision_root / 'metrics' / 'no_skill_reference_baselines.csv',
    revision_root / 'tables' / 'table_no_skill_reference_baselines.csv',
    revision_root / 'metrics' / 'baseline_summary.csv',
    revision_root / 'tables' / 'table_baselines.csv',
    revision_root / 'metrics' / 'component_check_summary.json',
    revision_root / 'tables' / 'table_component_checks.csv',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

optional_feature_outputs = [
    revision_root / 'metrics' / 'minirocket_only_baseline_summary.json',
    revision_root / 'predictions' / 'minirocket_only_oof_predictions.npz',
    revision_root / 'tables' / 'table_minirocket_only_class_metrics.csv',
    revision_root / 'tables' / 'table_minirocket_only_fold_summary.csv',
    revision_root / 'manifests' / 'minirocket_only_baseline_manifest.json',
    revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
    revision_root / 'metrics' / 'paired_full_vs_minirocket_bootstrap_samples.csv',
    revision_root / 'tables' / 'table_paired_full_vs_minirocket.csv',
    revision_root / 'manifests' / 'paired_full_vs_minirocket_manifest.json',
]
print('\nOptional feature-baseline outputs:')
for path in optional_feature_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print('\nNotebook 04 complete. Fair-baseline superiority remains blocked until all required fair comparator rows are complete under the same frozen OOF protocol.')
